# 01 – Data loading (Day 1)

| Column | Role |
|---|---|
| Consumer complaint narrative | X: the text the model reads |
| Product | y: what we predict (the team to route to) |
| Date received | used for the time-based train/test split |
| Complaint ID | unique key for tracking rows, never a feature |

In [5]:
import warnings
from pathlib import Path

import pandas as pd

In [6]:
PATH = "../data/raw/CFPB.csv"
COLS = ["Consumer complaint narrative", "Product", "Date received", "Complaint ID"]

# The last row of the file is cut off mid-text. The python engine can skip it,
# and on_bad_lines="warn" makes pandas raise a warning for each skipped line.
# catch_warnings(record=True) collects those warnings into the list `caught`.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")  # don't hide repeated warnings
    df = pd.read_csv(PATH, usecols=COLS, engine="python", on_bad_lines="warn")

In [7]:
# Logger: keep only the "Skipping line" warnings and print them
skipped = [str(w.message).strip() for w in caught if str(w.message).startswith("Skipping line")]
for msg in skipped:
    print("BAD LINE:", msg)

# We know exactly 1 row is broken. 0 or more than 1 means the file changed, so stop.
assert len(skipped) == 1, f"Expected exactly 1 bad line (truncated last row), got {len(skipped)}"

BAD LINE: Skipping line 2901250: unexpected end of data


In [8]:
# Every complaint must have its own ID. .duplicated() marks repeats as True; .sum() counts them.
n_dupes = df["Complaint ID"].duplicated().sum()
print("Duplicate Complaint IDs:", n_dupes)
assert n_dupes == 0, f"{n_dupes} duplicate Complaint IDs found"

Duplicate Complaint IDs: 0


In [9]:
# Expect (2901248, 4): all rows except the broken one, and only our 4 columns
print("Shape:", df.shape)
df.head()

Shape: (2901248, 4)


,Date received,Product,Consumer complaint narrative,Complaint ID
0,07/06/24,Credit reporting or other personal consumer re...,NaN,9447068
1,04/17/25,Credit card,NaN,13044866
2,05/13/25,Credit reporting or other personal consumer re...,I was a victim of identity theft. There are in...,13488561
3,09/18/23,Credit reporting or other personal consumer re...,NaN,7564456
4,05/23/24,Credit reporting or other personal consumer re...,In accordance with the Fair Credit Reporting a...,9076742


In [10]:
# Length of each non-null narrative after removing outer spaces
narrative_lengths = df["Consumer complaint narrative"].dropna().str.strip().str.len()
narrative_lengths.describe()

count    904577.000000
mean        991.364298
std        1268.845431
min           9.000000
25%         332.000000
50%         662.000000
75%        1198.000000
max       35984.000000
Name: Consumer complaint narrative, dtype: float64

In [11]:
# Shortest 20 narratives (by length)
narrative_lengths.sort_values().head(20)

651312      9
788788     10
1721147    10
322056     10
508572     10
664360     10
979886     10
2330823    10
1905931    10
1808823    10
9012       10
2535278    10
1700512    10
547989     10
2602935    11
498127     12
2638661    12
1428320    12
2726111    12
1967435    12
Name: Consumer complaint narrative, dtype: int64

In [12]:
# Most common short (<= 20 chars) narratives: which are junk, which are real complaints?
(
    df["Consumer complaint narrative"]
    .dropna()
    .str.strip()
    .loc[lambda s: s.str.len().le(20)]
    .value_counts()
    .head(20)
)

Consumer complaint narrative
XX/XX/XXXX              12
XX/XX/XXXXXX/XX/XXXX     8
See attached             8
see attached             6
Account is not mine      4
Account is incorrect     3
IDENTITY THEFT           3
see attachments          3
Not my account           3
Debt not mine            2
wrong date of birth      2
VIOLATION OF FDCPA       2
No one helped me.        2
See Attached             2
Account not mine         2
I have no knowledge      2
SEE ATTACHMENTS          2
Identity was used        1
identity theft           1
This is not my debt      1
Name: count, dtype: int64

In [13]:
# Subtask 6: keep only rows with a usable narrative.
# A narrative is junk if:
#   A) it is NaN (no text at all)
#   B) it is only redaction, like "XX/XX/XXXX"
#   C) it is a short "see attached" pointer (30 chars or less)
# Short real complaints like "Account not mine" are KEPT.
# Production note: junk tickets go to a human, not the model.

rows_before = len(df)
text = df["Consumer complaint narrative"].str.strip()

is_nan = text.isna()
is_redacted_only = text.str.replace(r"[X/\s]", "", regex=True).str.len().eq(0)
is_attachment_pointer = text.str.len().le(30) & text.str.contains("see attach", case=False, na=False)

junk = is_nan | is_redacted_only | is_attachment_pointer
df_text = df[~junk].copy()

print(f"NaN: {is_nan.sum():,} | redacted only: {is_redacted_only.sum():,} | attachment pointer: {is_attachment_pointer.sum():,}")
print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {len(df_text):,} ({len(df_text) / rows_before:.1%} kept)")

NaN: 1,996,671 | redacted only: 273 | attachment pointer: 97
Rows before: 2,901,248
Rows after:  904,207 (31.2% kept)


In [14]:
# Why 30? Look at both sides of the line.
# Side 1: rows the rule REMOVED (should all be junk)
print(text[is_attachment_pointer].sample(10, random_state=0).tolist())

# Side 2: rows just ABOVE 30 chars that mention "attach" and were KEPT (should be real complaints)
kept_near_line = text[~junk & text.str.contains("attach", case=False) & text.str.len().between(31, 60)]
print(kept_near_line.head(10).tolist())

['See Attached', 'Please see attached file', 'Please see attached file.', 'please see attached file', 'Please see attached file', 'please see attached file', 'See attached', 'Please see attached file', 'Please see attached file', 'See attached']
['This was paid on XX/XX/2023, receipt is attached.', 'I have attached an FTC report with fraudulent inquiries', 'please refer to the attachments', 'Please help me, I have attached supporting documents.', 'This is my last hope, I have attached supporting documents.', 'Please help me, I have attached supporting correspondence', 'This is my last hope, I have attached supporting documents.', 'Please help me, I have attached supporting documents.', 'never received updated on my account. see attached document', 'I have attached a letter that will explain a lot better.']


In [15]:
product_count = df_text["Product"].nunique()
product_counts = df_text["Product"].value_counts()
product_percentages = df_text["Product"].value_counts(normalize=True) * 100

In [16]:
print(f"Number of unique products: {product_count}\n")
pd.DataFrame({"rows": product_counts, "percent": product_percentages.round(2)})

# Old/new name pairs (NOT merged yet, Day 4). The CFPB renamed them on 25 Aug 2023:
# 1. "Credit reporting, credit repair services, or other personal consumer reports"
#    vs "Credit reporting or other personal consumer reports"   (renamed)
# 2. "Credit card or prepaid card" vs "Credit card" + "Prepaid card"   (split)
# 3. "Payday loan, title loan, or personal loan"
#    vs "Payday loan, title loan, personal loan, or advance loan"   (renamed)

Number of unique products: 14



,rows,percent
Product,,
Credit reporting or other personal consumer reports,607924,67.23
"Credit reporting, credit repair services, or other personal consumer reports",75948,8.40
Debt collection,66984,7.41
Checking or savings account,38269,4.23
"Money transfer, virtual currency, or money service",35797,3.96
Credit card,30572,3.38
Mortgage,13467,1.49
Student loan,9415,1.04
Vehicle loan or lease,8574,0.95


In [17]:
# Exact format MM/DD/YY (approved plan): no guessing.
# errors="coerce" turns unparseable values into NaT so we can count them.
df_text["Date received"] = pd.to_datetime(df_text["Date received"], format="%m/%d/%y", errors="coerce")

n_failed = df_text["Date received"].isna().sum()
print(f"Dates that failed to parse: {n_failed}")
assert n_failed == 0, f"{n_failed} dates failed to parse"

print(f"Earliest: {df_text['Date received'].min().date()}")
print(f"Latest:   {df_text['Date received'].max().date()}")
# Coverage: "Consumer disputed?" was 100% empty because the CFPB stopped collecting it in 2017,
# and this data starts in 2023.

Dates that failed to parse: 0
Earliest: 2023-01-01
Latest:   2025-06-30


In [18]:
# raw/ is never modified; processed data goes to interim/
Path("../data/interim").mkdir(exist_ok=True)
df_text.to_parquet("../data/interim/complaints_text.parquet", index=False)

# Check: reload and confirm the date is still a date
check = pd.read_parquet("../data/interim/complaints_text.parquet")
print(check.shape)
print(check.dtypes)

(904207, 4)
Date received                   datetime64[us]
Product                                    str
Consumer complaint narrative               str
Complaint ID                             int64
dtype: object


In [19]:
removed = df[is_redacted_only | is_attachment_pointer]
print(removed["Consumer complaint narrative"].head(10).tolist())

['XX/XX/XXXX', 'XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX', 'XXXX XXXX XXXX XXXX XX/XX/XXXXXXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXXXX/XX/XXXX', 'please see attached file', 'see attached', 'Please see attached file.', 'XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XX/XX/XXXX XXXX XXXX XXXX XXXX XX/XX/XXXX', 'XXXX XXXX XX/XX/XXXX XXXX XX/XX/XXXX XXXX XX/XX/XXXX XXXX XXXX XX/XX/XXXX XXXXXXXX XXXX XXXX XXXXXXXX XXXX', 'XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXXXX/XX/XXXX', 'XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXX XXXXXX/XX/XXXX']


In [20]:
edge = text[text.str.contains("see attach", case=False) & text.str.len().between(31, 80)]
print(edge.sample(10, random_state=0).tolist())


['I am a victim of the Equifax data breach please see attachments below.', 'See attached documents, the personal information you are reporting are not mine.', 'I am a victim of the Equifax data breach please see attachments below.', 'I am a victim of the Equifax data breach please see attachments below.', "Account on ChexSystems doesn't belong to me, please see attach FTC Report", 'I am a victim of the Equifax data breach please see attachments below.', 'SEE ATTACHED FOR FRAUD, INACCURATE, AND UNVERIFIABLE ACCOUNT WITH XXXX XXXX', 'Please report correct information to consumer bureaus. See attached', 'I am a victim of the Equifax data breach please see attachments below.', 'I am a victim of the Equifax data breach please see attachments below.']
